In [ ]:
# Analisis exploratorio del test GAIT Arms

# Este cuaderno documenta los pasos para cargar y revisar los datos IMU registrados en BASE-SPINE, LEFT-HAND y RIGHT-HAND durante el test GAIT_ARMS.

In [18]:
# Librerias y ubicacion del archivo
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA_PATH = Path("imus_data_27_sep/20250929214525_19305147-dcbdb9cc-714f-479a-b822-fcf41664e940.json")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado en {DATA_PATH}")

In [19]:
# Carga y normalizacion del JSON
with DATA_PATH.open(encoding="utf-8") as f:
    raw = json.load(f)

patient = raw.get("patient", {})
test_type = raw.get("testType")

records = []
for entry in raw.get("imuData", []):
    device = entry.get("deviceId")
    ts = entry.get("timestamp")
    acc = entry.get("accelerometer", {})
    gyro = entry.get("gyroscope", {})
    records.append(
        {
            "device": device,
            "timestamp": ts,
            "acc_x": acc.get("x"),
            "acc_y": acc.get("y"),
            "acc_z": acc.get("z"),
            "gyro_x": gyro.get("x"),
            "gyro_y": gyro.get("y"),
            "gyro_z": gyro.get("z"),
        }
    )

imu_df = pd.DataFrame.from_records(records)
if imu_df.empty:
    raise ValueError("No se encontraron registros en imuData")
imu_df.sort_values(["device", "timestamp"], inplace=True)
imu_df.reset_index(drop=True, inplace=True)
imu_df["acc_mag"] = np.sqrt(imu_df["acc_x"] ** 2 + imu_df["acc_y"] ** 2 + imu_df["acc_z"] ** 2)
imu_df["gyro_mag"] = np.sqrt(imu_df["gyro_x"] ** 2 + imu_df["gyro_y"] ** 2 + imu_df["gyro_z"] ** 2)
imu_df.head()

,device,timestamp,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,acc_mag,gyro_mag
0,BASE-SPINE,429111,-0.460205,-0.833374,0.430786,-18.310550,46.875000,43.945310,1.044929,66.811168
1,BASE-SPINE,429138,-0.459106,-0.898926,0.463501,-1.525879,17.883300,47.302250,1.110711,50.592920
2,BASE-SPINE,429164,-0.477173,-0.835571,0.387329,-11.108400,8.422852,46.569820,1.037254,48.611615
3,BASE-SPINE,429190,-0.583130,-0.922241,0.527588,8.972168,-7.690430,19.714360,1.211989,22.984745
4,BASE-SPINE,429217,-0.337646,-1.082642,0.574097,76.171880,-14.343260,-4.577637,1.271104,77.645600


In [3]:
# Resumen del paciente y conteos globales
print(f"Paciente: {patient.get('name', 'N/D')} ({patient.get('id', 'N/D')})")
print(f"Fecha de nacimiento: {patient.get('birthDate', 'N/D')}")
print(f"Observaciones: {patient.get('observations') or 'Sin observaciones registradas'}")
print(f"Tipo de prueba: {test_type}")
print(f"Total de registros IMU: {imu_df.shape[0]:,}")
print(f"Dispositivos disponibles: {sorted(imu_df['device'].unique().tolist())}")

Paciente: Luis Alberto Londoño (19305147)
Fecha de nacimiento: 02/04/1956
Observaciones: Sin observaciones registradas
Tipo de prueba: GAIT_ARMS
Total de registros IMU: 3,605
Dispositivos disponibles: ['BASE-SPINE', 'LEFT-HAND', 'RIGHT-HAND']


In [5]:
# Brecha temporal y frecuencia de muestreo aproximada
def summarize_sampling(df: pd.DataFrame) -> pd.DataFrame:
    stats = []
    for device, grp in df.groupby("device"):
        timestamps = grp["timestamp"].astype(float).sort_values()
        deltas = timestamps.diff().dropna()
        stats.append(
            {
                "device": device,
                "samples": len(grp),
                "t_start": timestamps.iloc[0],
                "t_end": timestamps.iloc[-1],
                "duration_ms": timestamps.iloc[-1] - timestamps.iloc[0],
                "mean_dt": deltas.mean() if not deltas.empty else np.nan,
                "median_dt": deltas.median() if not deltas.empty else np.nan,
                "min_dt": deltas.min() if not deltas.empty else np.nan,
                "max_dt": deltas.max() if not deltas.empty else np.nan,
            }
        )
    return pd.DataFrame(stats)

sampling_summary = summarize_sampling(imu_df)
sampling_summary = sampling_summary.assign(
    duration_s=lambda d: d["duration_ms"] / 1000,
    mean_hz=lambda d: 1000 / d["mean_dt"].replace({0: np.nan}),
    median_hz=lambda d: 1000 / d["median_dt"].replace({0: np.nan}),
)
sampling_summary

,device,samples,t_start,t_end,duration_ms,mean_dt,median_dt,min_dt,max_dt,duration_s,mean_hz,median_hz
0,BASE-SPINE,1343,429111.0,464569.0,35458.0,26.421759,26.0,26.0,85.0,35.458,37.847594,38.461538
1,LEFT-HAND,980,341232.0,375270.0,34038.0,34.768131,36.0,26.0,90.0,34.038,28.761972,27.777778
2,RIGHT-HAND,1282,553675.0,587357.0,33682.0,26.293521,26.0,26.0,52.0,33.682,38.032183,38.461538


In [6]:
# Estadisticas descriptivas por dispositivo
metric_cols = [
    "acc_x",
    "acc_y",
    "acc_z",
    "acc_mag",
    "gyro_x",
    "gyro_y",
    "gyro_z",
    "gyro_mag",
]
desc = imu_df.groupby("device")[metric_cols].agg(["mean", "std", "min", "max"])
desc.columns = [f"{col}_{stat}" for col, stat in desc.columns]
desc.reset_index()

,device,acc_x_mean,acc_x_std,acc_x_min,acc_x_max,acc_y_mean,acc_y_std,acc_y_min,acc_y_max,acc_z_mean,...,gyro_y_min,gyro_y_max,gyro_z_mean,gyro_z_std,gyro_z_min,gyro_z_max,gyro_mag_mean,gyro_mag_std,gyro_mag_min,gyro_mag_max
0,BASE-SPINE,0.278034,0.548991,-1.128784,1.372437,0.002039,0.334799,-1.267212,1.074341,0.661230,...,-268.55470,276.18410,1.500065,56.671303,-472.90040,208.80130,60.628328,64.940043,0.390816,517.105500
1,LEFT-HAND,0.125187,0.125395,-0.066406,0.258301,0.411170,0.133438,0.214722,0.583740,0.887123,...,-18.37158,16.35742,0.216986,1.860197,-15.62500,13.91602,2.038171,4.036622,0.105716,47.614696
2,RIGHT-HAND,0.010860,0.150146,-0.233154,0.158813,-0.496312,0.194231,-0.756226,-0.189331,0.860406,...,-15.62500,29.60205,0.430626,2.134746,-14.03809,25.81787,2.065326,4.702352,0.000000,60.284509


In [8]:
# Valores faltantes por dispositivo y variable
missing = (
    imu_df.drop(columns=["device"])
    .isna()
    .groupby(imu_df["device"])
    .sum()
    .reset_index()
    .rename(columns={"index": "device"})
)
missing

,device,timestamp,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,acc_mag,gyro_mag
0,BASE-SPINE,0,0,0,0,0,0,0,0,0
1,LEFT-HAND,0,0,0,0,0,0,0,0,0
2,RIGHT-HAND,0,0,0,0,0,0,0,0,0


In [9]:
# Serie temporal de la magnitud de aceleracion
fig_acc = px.line(
    imu_df,
    x="timestamp",
    y="acc_mag",
    color="device",
    title="Magnitud de aceleracion por dispositivo",
    labels={"timestamp": "timestamp (ms)", "acc_mag": "|a| (g)"},
)
fig_acc

In [10]:
# Serie temporal de la magnitud de velocidad angular
fig_gyro = px.line(
    imu_df,
    x="timestamp",
    y="gyro_mag",
    color="device",
    title="Magnitud de velocidad angular por dispositivo",
    labels={"timestamp": "timestamp (ms)", "gyro_mag": "|w| (deg/s)"},
)
fig_gyro

In [11]:
# Componentes individuales del acelerometro
acc_long = imu_df.melt(
    id_vars=["device", "timestamp"],
    value_vars=["acc_x", "acc_y", "acc_z"],
    var_name="axis",
    value_name="acc"
)
fig_acc_axis = px.line(
    acc_long,
    x="timestamp",
    y="acc",
    color="device",
    facet_row="axis",
    title="Componentes del acelerometro",
    labels={"acc": "aceleracion (g)", "timestamp": "timestamp (ms)"},
)
fig_acc_axis.update_layout(showlegend=True)
fig_acc_axis

In [14]:
# Componentes individuales del giroscopio
gyro_long = imu_df.melt(
    id_vars=["device", "timestamp"],
    value_vars=["gyro_x", "gyro_y", "gyro_z"],
    var_name="axis",
    value_name="gyro"
)
fig_gyro_axis = px.line(
    gyro_long,
    x="timestamp",
    y="gyro",
    color="device",
    facet_row="axis",
    title="Componentes del giroscopio",
    labels={"gyro": "velocidad angular (deg/s)", "timestamp": "timestamp (ms)"},
)
fig_gyro_axis.update_layout(showlegend=True)
fig_gyro_axis

## Notas y proximos pasos
- Revisar ventanas de interes en las señales crudas para detectar artefactos o segmentos relevantes dle test.
- Aplicar filtrado o suavizado adicional antes de extraer indicadores clinicos (por ejemplo, gait speed, balanceo de brazos).
- Comparar este registro con otras sesiones del paciente para evaluar consistencia y progresion.